# Домашнее задание №1

Посчитать количество рейтингов больше 4 для каждого фильма и вывести фильмы в порядке убывания количества этих оценок.

In [1]:
import os
from collections import defaultdict, namedtuple
from functools import reduce
from joblib import Parallel, delayed

In [2]:
with open(os.path.join("..", "data", "user_ratedmovies.dat"), "rb") as f:
    data = f.readlines()
headers = data[0].decode().split("\t")[:3]
data = [row.decode().split("\t")[:3] for row in data[1:]]

In [3]:
headers

['userID', 'movieID', 'rating']

In [4]:
data[0]

['75', '3', '1']

In [5]:
len(data)

855598

Пишем `map`, `shuffle` и `reduce` + параллелим вычисления. Лучше задавать `batch_size` при распараллеливании, либо даже заранее всё разбить на батчи, будет быстрее.

Также посмотрите на то, нет ли перекоса в данных после `shuffle`, можете попробовать использовать остаток от деления не простого hash, а ввести какую-то функию.

In [6]:
PARTITION_SIZE = 1024

partitions = [data[i:i+PARTITION_SIZE] for i in range(0, len(data), PARTITION_SIZE)]

In [7]:
Row = namedtuple("Row", ["movie_id", "rating"])


def map_shuffle(partition: list[list[str]], n_nodes: int) -> defaultdict[int, list[Row]]:
    return shuffle_rating(mapper_rating(partition), n_nodes)


def mapper_rating(partition: list[list[str]]) -> list[Row]:
    return [Row(row[1], float(row[2])) for row in partition]


def shuffle_rating(mapper_result: list[Row], n_nodes: int = 5) -> defaultdict[int, list[Row]]:
    shuffled_data = [(int(row.movie_id) % n_nodes, row) for row in mapper_result]
    return create_chunks(shuffled_data)


def create_chunks(shuffled_data: list[tuple[int, Row]]) -> defaultdict[int, list[Row]]:
    result = defaultdict(list)
    for idx, data in shuffled_data:
        result[idx].append(data)
    return result

In [8]:
N_NODES = 5


with Parallel(n_jobs=4, verbose=10) as parallel:
    answer1 = parallel(delayed(map_shuffle)(partition, N_NODES) for partition in partitions)

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Batch computation too fast (0.18616732558762214s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Batch computation too fast (0.026567697525024414s.) Setting batch_size=4.
[Parallel(n_jobs=4)]: Done  28 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Batch computation too fast (0.04190850257873535s.) Setting batch_size=8.
[Parallel(n_jobs=4)]: Done  56 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Batch computation too fast (0.08046078681945801s.) Setting batch_size=16.
[Parallel(n_jobs=4)]: Done 116 tasks      | elapsed:    0.8s
[Parallel(n_jobs=4)]: Batch computation too fast (0.1470794677734375s.) Setting batch_size=32.
[Parallel(n_jobs=4)]: Done 292 tasks      | elapsed:    1.2s
[Parallel(n_jobs=4)]:

In [9]:
shuffle_stage = {i:[] for i in range(N_NODES)}
for partition in answer1:
    for idx, rows in partition.items():
        shuffle_stage[idx].extend(rows)

In [10]:
for idx, rows in shuffle_stage.items():
    print(f"{idx}: number of rows = {len(rows)}")

0: number of rows = 170211
1: number of rows = 169817
2: number of rows = 170990
3: number of rows = 183481
4: number of rows = 161099


In [11]:
def reduce_rating(rows: list[Row]) -> list[tuple[str, float, int]]:
    ratings = defaultdict(float)
    counter = defaultdict(int)
    for row in rows:
        ratings[row.movie_id] += row.rating
        counter[row.movie_id] += 1
    return [
        (movie_id, ratings[movie_id] / counter[movie_id], counter[movie_id])
        for movie_id in ratings
    ]

In [12]:
with Parallel(n_jobs=4, verbose=10) as parallel:
    answer2 = parallel(delayed(reduce_rating)(partition) for partition in shuffle_stage.values())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   2 out of   5 | elapsed:    1.2s remaining:    1.8s
[Parallel(n_jobs=4)]: Done   3 out of   5 | elapsed:    1.6s remaining:    1.0s
[Parallel(n_jobs=4)]: Done   5 out of   5 | elapsed:    2.3s finished


После `reduce` всё можно собрать в одном цикле, считаем, что данные переслали после на 1 машину и агрегируем.

In [13]:
result = []
for partition in answer2:
    result.extend(partition)

In [14]:
for movie_id, avg_rating, count in sorted(
    filter(lambda x: x[1] > 4.0, result), key=lambda x: x[2], reverse=True
)[:20]:
    print(f"{movie_id=}: {avg_rating=:.3}, {count=}")

movie_id='2571': avg_rating=4.17, count=1670
movie_id='4993': avg_rating=4.08, count=1576
movie_id='296': avg_rating=4.24, count=1537
movie_id='5952': avg_rating=4.03, count=1528
movie_id='2858': avg_rating=4.11, count=1472
movie_id='7153': avg_rating=4.09, count=1457
movie_id='318': avg_rating=4.37, count=1441
movie_id='2959': avg_rating=4.25, count=1434
movie_id='593': avg_rating=4.05, count=1431
movie_id='260': avg_rating=4.08, count=1365
movie_id='4226': avg_rating=4.18, count=1320
movie_id='1196': avg_rating=4.1, count=1306
movie_id='1198': avg_rating=4.14, count=1304
movie_id='50': avg_rating=4.29, count=1218
movie_id='47': avg_rating=4.06, count=1206
movie_id='7361': avg_rating=4.1, count=1202
movie_id='527': avg_rating=4.17, count=1170
movie_id='32': avg_rating=4.01, count=1158
movie_id='1136': avg_rating=4.13, count=1151
movie_id='4973': avg_rating=4.19, count=1137
